In [2]:
# Cell 1 - Libraries
!pip install requests -q
import requests
import pandas as pd
import time
import os


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
# Cell 2 - Fixed GDELT Function
def verify_gdelt(keywords, year='2020'):
    query = ' '.join(keywords[:3])
    url = (
        f'https://api.gdeltproject.org/api/v2/doc/doc'
        f'?query={query}&mode=artlist&maxrecords=5'
        f'&format=json'
        f'&startdatetime={year}0101000000'
        f'&enddatetime={year}1231235959'
    )
    for attempt in range(3):
        try:
            r = requests.get(url, timeout=20, verify=False)
            return r.json().get('articles', [])
        except Exception as e:
            print(f"  Attempt {attempt+1} failed — retrying...")
            time.sleep(3)
    return []

In [ ]:
# Cell 3 - Testing
import warnings
warnings.filterwarnings('ignore')

test_topics = {
    'Topic 0': ['trump', 'election', 'president'],
    'Topic 1': ['health', 'covid', 'hospital'],
    'Topic 2': ['climate', 'environment', 'carbon'],
    'Topic 3': ['economy', 'market', 'business'],
}

results = []
for name, keywords in test_topics.items():
    print(f"Checking: {name}...")
    arts = verify_gdelt(keywords, year='2020')
    status = 'VERIFIED' if arts else 'NOT FOUND'
    print(f"  {status} — {len(arts)} matches")
    results.append({
        'Topic'   : name,
        'Keywords': str(keywords),
        'Matches' : len(arts),
        'Status'  : status
    })
    time.sleep(3)

pd.DataFrame(results)

Checking: Topic 0...
  VERIFIED — 5 matches
Checking: Topic 1...
  Attempt 1 failed — retrying...
  Attempt 2 failed — retrying...
  Attempt 3 failed — retrying...
  NOT FOUND — 0 matches
Checking: Topic 2...
  Attempt 1 failed — retrying...
  Attempt 2 failed — retrying...
  Attempt 3 failed — retrying...
  NOT FOUND — 0 matches
Checking: Topic 3...
  Attempt 1 failed — retrying...
  Attempt 2 failed — retrying...
  Attempt 3 failed — retrying...
  NOT FOUND — 0 matches


,Topic,Keywords,Matches,Status
0,Topic 0,"['trump', 'election', 'president']",5,VERIFIED
1,Topic 1,"['health', 'covid', 'hospital']",0,NOT FOUND
2,Topic 2,"['climate', 'environment', 'carbon']",0,NOT FOUND
3,Topic 3,"['economy', 'market', 'business']",0,NOT FOUND


In [11]:
os.makedirs('../reports', exist_ok=True)
pd.DataFrame(results).to_csv('../reports/gdelt_results.csv', index=False)
print("Saved: ../reports/gdelt_results.csv")

Saved: ../reports/gdelt_results.csv
